# Data Cleaning and Visualization Project

This notebook cleans the employee dataset, exports `cleaned_data.csv`, and creates four visualizations.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

FILE_NAME = 'sample_data_cleaning_project - Sample_data_cleaning_project.csv'
OUTPUT_FILE = 'cleaned_data.csv'
if not os.path.isfile(FILE_NAME):
    raise FileNotFoundError(FILE_NAME)
data = pd.read_csv(FILE_NAME)
print(f'Loaded {data.shape[0]} rows and {data.shape[1]} columns')
display(data.head())

In [ ]:
print('Missing values before cleaning:')
print(data.isna().sum())

for column in ['Name', 'Department']:
    data[column] = data[column].astype('string').str.strip().str.lower()
data['Age'] = pd.to_numeric(data['Age'], errors='coerce')
data['Salary'] = pd.to_numeric(data['Salary'], errors='coerce')
data['Age'] = data['Age'].fillna(data['Age'].median())
data['Salary'] = data['Salary'].fillna(data['Salary'].median())
data['Join_Date'] = pd.to_datetime(data['Join_Date'], errors='coerce')
data = data.dropna(subset=['Join_Date']).copy()
print('Missing values after treatment:')
print(data.isna().sum())

In [ ]:
duplicates_removed = int(data.duplicated().sum())
data = data.drop_duplicates().copy()
q1, q3 = data['Salary'].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers_removed = int(((data['Salary'] < lower) | (data['Salary'] > upper)).sum())
data = data.loc[data['Salary'].between(lower, upper)].copy()
data['Age'] = data['Age'].round().astype(int)
data = pd.get_dummies(data, columns=['Department'], prefix='Department', drop_first=False, dtype=int)
data.to_csv(OUTPUT_FILE, index=False)
print(f'Duplicates removed: {duplicates_removed}')
print(f'Salary outliers removed: {outliers_removed}')
print(f'Saved {OUTPUT_FILE} with shape {data.shape}')
assert data.notna().all().all()
assert not data.duplicated().any()

In [ ]:
cleaned_data = pd.read_csv(OUTPUT_FILE)
department_columns = [c for c in cleaned_data if c.startswith('Department_')]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
cleaned_data[department_columns].sum().rename(lambda x: x.replace('Department_', '')).plot(kind='bar', ax=axes[0, 0], title='Department Distribution')
axes[0, 0].set_ylabel('Employees')
axes[0, 1].hist(cleaned_data['Salary'], bins=10, edgecolor='black')
axes[0, 1].set(title='Salary Distribution', xlabel='Salary', ylabel='Frequency')
axes[1, 0].scatter(cleaned_data['Age'], cleaned_data['Salary'])
axes[1, 0].set(title='Salary vs. Age', xlabel='Age', ylabel='Salary')
for column in department_columns:
    mask = cleaned_data[column].eq(1)
    axes[1, 1].scatter(cleaned_data.loc[mask, 'Salary'], cleaned_data.loc[mask, 'Age'], label=column.replace('Department_', ''))
axes[1, 1].set(title='Salary vs. Age by Department', xlabel='Salary', ylabel='Age')
axes[1, 1].legend(title='Department')
for ax in axes.flat: ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()